In [1]:
from __future__ import annotations
import numpy as np
from dataclasses import dataclass
from typing import Literal, Tuple, Dict
from sklearn.decomposition import NMF

In [2]:
_EPS = 1e-12

def _safe(x: np.ndarray, eps: float = _EPS) -> np.ndarray:
    """Replace nonpositive entries by small epsilon for logs / divisions."""
    y = x.copy()
    y[y <= 0] = eps
    return y

def normalize_joint(P: np.ndarray) -> np.ndarray:
    """Return joint probability p(π,i) from a nonnegative matrix P."""
    s = P.sum()
    if s <= 0:
        raise ValueError("Input matrix must have positive sum.")
    return P / s

def normalize_cols(M: np.ndarray) -> np.ndarray:
    """Column-wise normalize to sum 1 (each column is a conditional distribution)."""
    colsum = M.sum(axis=0, keepdims=True)
    colsum[colsum <= 0] = 1.0
    return M / colsum

def entropy(p: np.ndarray, base: float = np.e) -> float:
    """Shannon entropy of a flat vector probability distribution."""
    p = _safe(p.ravel())
    p = p / p.sum()
    logf = np.log if base == np.e else (lambda x: np.log(x)/np.log(base))
    return float(-(p * logf(p)).sum())

def mutual_information(P: np.ndarray, base: float = np.e) -> float:
    """I(X;Y) for joint P (shape NxM)."""
    P = normalize_joint(P)
    px = _safe(P.sum(axis=1, keepdims=True))
    py = _safe(P.sum(axis=0, keepdims=True))
    denom = px @ py
    mask = P > 0
    logf = np.log if base == np.e else (lambda x: np.log(x)/np.log(base))
    I = (P[mask] * (logf(P[mask]) - logf(denom[mask]))).sum()
    return float(I)

def conditional_entropy_from_joint(P: np.ndarray, base: float = np.e) -> float:
    """
    H(Π|I) for joint p(π,i) (Π rows, I columns).
    H(Π|I) = Σ_i p(i) H(Π|i) = - Σ_{π,i} p(π,i) log p(π|i).
    """
    P = normalize_joint(P)
    p_cond = normalize_cols(P)
    mask = P > 0
    logf = np.log if base == np.e else (lambda x: np.log(x)/np.log(base))
    return float(-(P[mask] * logf(_safe(p_cond)[mask])).sum())

In [3]:
def fit_nmf(
    data: np.ndarray,
    rank: int,
    seed: int = 42,
    loss: Literal["frob","kull"] = "frob",
    max_iter: int = 10000,
    tol: float = 1e-4,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Fit NMF to nonnegative 'data' of shape (M_images, N_pixels) or (N, M).
    We assume data rows are images, columns are pixels as in sklearn.datasets.
    Returns (W, H) where data ≈ W @ H, W≥0 (M×R), H≥0 (R×N).
    """
    if loss == "frob":
        model = NMF(
            n_components=rank, init="random", random_state=seed,
            max_iter=max_iter, tol=tol
        )
    else:
        model = NMF(
            n_components=rank, init="random", random_state=seed,
            solver="mu", beta_loss="kullback-leibler",
            max_iter=max_iter, tol=tol
        )
    W = model.fit_transform(data)
    H = model.components_
    return W, H

In [4]:
@dataclass
class ProbPack:
    p_pi: np.ndarray     
    phat_pi: np.ndarray    
    p_pi_marg: np.ndarray  
    p_i_marg: np.ndarray  
    p_b: np.ndarray      
    p_bi: np.ndarray     
    p_pib: np.ndarray    
    p_pi_cond: np.ndarray  
    phat_pi_cond: np.ndarray  


def build_probabilities_from_BW(
    P: np.ndarray,         
    B: np.ndarray,         
    W: np.ndarray,         
    eps: float = 1e-12,
    prune_empty: bool = True,
) -> ProbPack:
    """
    Numerically-stable construction of all probabilities used in Appendix A.
    - Uses float64
    - Broadcasts instead of diag-matmul
    - Prunes components with near-zero mass (optional)
    - Renormalizes to keep marginals exact and arrays finite
    """
    P = np.asarray(P, dtype=np.float64).copy()
    B = np.asarray(B, dtype=np.float64).copy()
    W = np.asarray(W, dtype=np.float64).copy()

    p = normalize_joint(P)   
    N, M = p.shape

    p_i = p.sum(axis=0)     
    p_pi_marg = p.sum(axis=1)   

    denom_B = B.sum(axis=0, keepdims=True)  
    denom_B[denom_B <= 0] = 1.0
    p_pi_given_b = B / denom_B   

    sumB = denom_B.ravel()     
    p_bi = (W * sumB[:, None])   

    total = p_bi.sum()
    if not np.isfinite(total) or total <= 0:
        p_bi = np.full_like(p_bi, 1.0 / (p_bi.size))
    else:
        p_bi = p_bi / total

    colsum = p_bi.sum(axis=0, keepdims=True)  
    empty_cols = (colsum <= eps)
    if np.any(empty_cols):
        p_bi[:, empty_cols[0]] = 1.0 / p_bi.shape[0]
        colsum = p_bi.sum(axis=0, keepdims=True)
    p_bi = p_bi / colsum * p_i[None, :]
    p_bi = p_bi / max(p_bi.sum(), eps)

    p_b = p_bi.sum(axis=1)            
    if prune_empty:
        keep = p_b > 1e-14
        if keep.sum() == 0:
            keep = np.ones_like(p_b, dtype=bool)
            keep[np.argmax(p_b)] = True
        if keep.sum() < len(p_b):
            p_pi_given_b = p_pi_given_b[:, keep]
            p_bi = p_bi[keep, :]
            p_b = p_b[keep]
            p_bi = p_bi / max(p_bi.sum(), eps)
            p_b = p_bi.sum(axis=1)

    p_pib = p_pi_given_b * p_b[None, :]        
    p_pib_sum = p_pib.sum()
    if p_pib_sum <= 0 or not np.isfinite(p_pib_sum):
        p_pib = (np.ones((N, 1)) / N) @ p_b[None, :]
    else:
        p_pib = p_pib / p_pib_sum
        colsum_pb = p_pib.sum(axis=0, keepdims=True) 
        colsum_pb[colsum_pb <= eps] = 1.0
        p_pib = p_pib / colsum_pb * p_b[None, :]
        p_pib = p_pib / max(p_pib.sum(), eps)

    phat = p_pi_given_b @ p_bi                    
    if not np.all(np.isfinite(phat)):
        # robust fallback
        phat = np.einsum('nb,bi->ni', _safe(p_pi_given_b), _safe(p_bi))
    s_phat = phat.sum()
    if s_phat <= 0 or not np.isfinite(s_phat):
        phat = p.copy()
    else:
        phat = phat / s_phat

    phat_col = phat.sum(axis=0, keepdims=True)   
    phat_col[phat_col <= eps] = 1.0
    phat = phat / phat_col * p_i[None, :]
    phat_row = phat.sum(axis=1, keepdims=True)  
    phat_row[phat_row <= eps] = 1.0
    phat = phat / phat_row * p_pi_marg[:, None]
    phat = phat / max(phat.sum(), eps)

    p_pi_cond = normalize_cols(p)               
    phat_pi_cond = normalize_cols(phat)     

    return ProbPack(
        p_pi=p,
        phat_pi=phat,
        p_pi_marg=p_pi_marg,
        p_i_marg=p_i,
        p_b=p_b,
        p_bi=p_bi,
        p_pib=p_pib,
        p_pi_cond=p_pi_cond,
        phat_pi_cond=phat_pi_cond
    )

In [5]:
def I_B_I(prob: ProbPack, base: float = np.e) -> float:
    """I(𝔅:𝓘) from p(b,i)."""
    return mutual_information(prob.p_bi, base=base)

def I_B_Pi(prob: ProbPack, base: float = np.e) -> float:
    """I(𝔅:Π) from p(π,b)."""
    return mutual_information(prob.p_pib, base=base)

def I_Pi_I(prob: ProbPack, base: float = np.e) -> float:
    """I(Π:𝓘) from data p(π,i)."""
    return mutual_information(prob.p_pi, base=base)

def Ihat_Pi_I(prob: ProbPack, base: float = np.e) -> float:
    """Ĩ(Π:𝓘) from \hat p(π,i) but with data marginals (as in Appendix)."""
    phat = normalize_joint(prob.phat_pi)
    px = _safe(prob.p_pi_marg.reshape(-1, 1))
    py = _safe(prob.p_i_marg.reshape(1, -1))
    denom = px @ py
    mask = phat > 0
    logf = np.log if base == np.e else (lambda x: np.log(x)/np.log(base))
    return float((phat[mask] * (logf(phat[mask]) - logf(denom[mask]))).sum())

def H_Pi_given_I(prob: ProbPack, base: float = np.e) -> float:
    """H(Π|𝓘) from data p(π,i)."""
    return conditional_entropy_from_joint(prob.p_pi, base=base)

def Hhat_Pi_given_I(prob: ProbPack, base: float = np.e) -> float:
    """Ĥ(Π|𝓘) from \hat p(π,i) but with same p(i) (holds anyway)."""
    return conditional_entropy_from_joint(prob.phat_pi, base=base)

def H_B(prob: ProbPack, base: float = np.e) -> float:
    return entropy(prob.p_b, base=base)

def H_I(prob: ProbPack, base: float = np.e) -> float:
    return entropy(prob.p_i_marg, base=base)

def H_Pi(prob: ProbPack, base: float = np.e) -> float:
    return entropy(prob.p_pi_marg, base=base)

def NMI_tilde(Ixy: float, Hx: float, Hy: float) -> float:
    """~I(X:Y) = I / min(H(X), H(Y))  (0..1)."""
    denom = max(min(Hx, Hy), _EPS)
    return float(Ixy / denom)

def NMI_circle(Ixy: float, Hx: float, Hy: float) -> float:
    """I°(X:Y) = 2I / (H(X)+H(Y))  (0..1)."""
    denom = max(Hx + Hy, _EPS)
    return float(2.0 * Ixy / denom)


In [6]:
def relation(a: float, b: float, tol: float = 1e-8) -> str:
    """Return relation symbol comparing a ? b with tolerance."""
    if a > b + tol:
        return ">"
    elif b > a + tol:
        return "<"
    else:
        return "≈"

def check_appendix_relations(
    P_images_by_pixels: np.ndarray,
    rank: int,
    seed: int = 42,
    loss: Literal["frob","kull"] = "frob",
    base: float = np.e,
    tol: float = 1e-8,
    transpose_input: bool = True,
    report: bool = True,
) -> Dict[str, float]:
    """
    Given raw nonnegative data (M_images × N_pixels) by default, compute:
      - Fit NMF
      - Build probabilities (treating matrix as P[π,i] with shape N×M)
      - Compute/compare all '?' equations.
    If your matrix is already (N_pixels × M_images), set transpose_input=False.
    """
    P_raw = P_images_by_pixels
    if transpose_input:
        P = P_raw.T 
    else:
        P = P_raw.copy()

    W, H = fit_nmf(P.T, rank=rank, seed=seed, loss=loss) 

    B = H.T  
    Wp = W.T  

    prob = build_probabilities_from_BW(P, B, Wp)

    IBI  = I_B_I(prob, base=base)
    IBPi = I_B_Pi(prob, base=base)
    IPiI = I_Pi_I(prob, base=base)
    Ihat = Ihat_Pi_I(prob, base=base)

    Hcond   = H_Pi_given_I(prob, base=base)
    Hhat    = Hhat_Pi_given_I(prob, base=base)
    HB      = H_B(prob, base=base)
    HI      = H_I(prob, base=base)
    HPi     = H_Pi(prob, base=base)

    Ntilde_BI  = NMI_tilde(IBI,  HB, HI)
    Ntilde_BPi = NMI_tilde(IBPi, HB, HPi)
    Ncirc_BI   = NMI_circle(IBI,  HB, HI)
    Ncirc_BPi  = NMI_circle(IBPi, HB, HPi)

    if report:
        print(f"\n=== Appendix A checks (rank={rank}, loss={loss}, seed={seed}) ===")
        print(f"min(I(B:I), I(B:Π)) {relation(min(IBI, IBPi), IPiI, tol)} I(Π:I)"
              f"   =>  min({IBI:.6f}, {IBPi:.6f}) = {min(IBI, IBPi):.6f} ? {IPiI:.6f}")
        print(f"I(Π:I) {relation(IPiI, Ihat, tol)} Ĩ(Π:I)"
              f"                 =>  {IPiI:.6f} ? {Ihat:.6f}")
        print(f"H(Π|I) {relation(Hcond, Hhat, tol)} Ĥ(Π|I)"
              f"                 =>  {Hcond:.6f} ? {Hhat:.6f}")
        print(f"~I(B:I) {relation(Ntilde_BI, Ntilde_BPi, tol)} ~I(B:Π)"
              f"               =>  {Ntilde_BI:.6f} ? {Ntilde_BPi:.6f}")
        print(f"I°(B:I) {relation(Ncirc_BI, Ncirc_BPi, tol)} I°(B:Π)"
              f"               =>  {Ncirc_BI:.6f} ? {Ncirc_BPi:.6f}")

    return dict(
        I_BI=IBI,
        I_BPi=IBPi,
        I_PiI=IPiI,
        Ihat_PiI=Ihat,
        H_Pi_given_I=Hcond,
        Hhat_Pi_given_I=Hhat,
        H_B=HB, H_I=HI, H_Pi=HPi,
        Ntilde_BI=Ntilde_BI, Ntilde_BPi=Ntilde_BPi,
        Ncirc_BI=Ncirc_BI, Ncirc_BPi=Ncirc_BPi
    )


In [7]:
# from sklearn.datasets import fetch_olivetti_faces

In [8]:
# faces = fetch_olivetti_faces()

In [9]:
# res = check_appendix_relations(
#     P_images_by_pixels=faces.data,
#     rank=12,
#     seed=48,
#     loss="frob",   # or "kull"
#     base=np.e,
#     tol=1e-9,
#     transpose_input=True
# )